# 🔍 模組活動三｜環境化學解讀與 AI 查證
### 把圖表與數據交給 AI 推論成因 → 你當「科學大法官」逐條查證，揪出 AI 幻覺
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rlirdo/envchem-ai-module-kit/blob/main/notebooks/M3_interpret_verify.ipynb)　｜　[懶人包網站](https://rlirdo.github.io/envchem-ai-module-kit/)　｜　[本模組操作手冊](https://rlirdo.github.io/envchem-ai-module-kit/m3.html)　｜　[Prompt 指南](https://rlirdo.github.io/envchem-ai-module-kit/prompts.html)

| 步驟 | 你要做的事 | 成功的樣子 |
|---|---|---|
| 1 | 產生「數據摘要」與圖表，餵給 AI | AI 給出化學解釋 |
| 2 | 把 AI 的回答拆成一條一條「主張」 | 主張清單 5–8 條 |
| 3 | 逐條查證：課本、測站背景、文獻、NotebookLM | 每條都有依據 |
| 4 | 填「AI 推論查證表」 | ✔ 正確／△ 部分／✘ 錯誤 |
| 5 | 修正論述＋AI 使用反思 | 一段正確的化學解釋 |

<sub>NDHU 自資系 游豐兆 博士生製作｜指導教授 楊悠娟 副教授｜國立東華大學 115-1 環境化學 × 生成式 AI 融入專業課程計畫</sub>

In [1]:
# 載入懶人包的「環境部 API 常用代碼庫」（只做一次）
import os, urllib.request
if not os.path.exists('moenv_api.py'):
    urllib.request.urlretrieve('https://rlirdo.github.io/envchem-ai-module-kit/moenv_api.py', 'moenv_api.py')
import moenv_api as ma
import pandas as pd
print('✅ 代碼庫已載入：', list(ma.DATASETS))

✅ 代碼庫已載入： ['aqx_p_432', 'aqx_p_488', 'aqx_p_13', 'aqx_p_07']


In [2]:
#@title ▶ 執行：載入資料並算 24 小時平均（可直接執行）
df = pd.read_csv('https://raw.githubusercontent.com/rlirdo/envchem-ai-module-kit/main/data/hualien_aqx_p_488_hourly_20260917_20261017.csv', dtype=str)       # 也可以換成你自己 hualien_clean.csv
clean, _ = ma.clean_numeric(df, ['o3', 'no2', 'nox', 'no', 'pm2.5', 'windspeed'])
clean['datacreationdate'] = pd.to_datetime(clean['datacreationdate'])
clean['hour'] = clean['datacreationdate'].dt.hour
di = clean.groupby('hour')[['o3', 'no2', 'nox', 'pm2.5', 'windspeed']].mean().round(1)
print('期間：', clean['datacreationdate'].min(), '～', clean['datacreationdate'].max(), '，共', len(clean), '筆')
di

期間： 2026-09-17 18:00:00 ～ 2026-10-17 17:00:00 ，共 718 筆


,o3,no2,nox,pm2.5,windspeed
hour,,,,,
0,32.1,2.4,3.4,9.1,1.6
1,28.6,2.3,3.3,8.3,1.7
2,26.3,1.9,2.9,9.1,1.5
3,24.9,1.6,2.6,8.8,1.7
4,24.7,1.3,2.2,8.1,1.7
5,23.3,1.3,2.2,7.8,1.8
6,21.0,1.6,2.6,8.0,1.8
7,19.2,2.2,3.4,8.3,2.0
8,20.5,2.3,3.9,8.3,1.9


## 步驟 1｜產生「給 AI 看的數據摘要」
AI 看不懂你的 Colab 變數，要把**重點數字**整理成文字再貼給它（也可以上傳步驟 4 的圖）。

In [3]:
#@title ▶ 執行：產生數據摘要文字（複製去問 AI）
pk_o3, lo_o3 = di['o3'].idxmax(), di['o3'].idxmin()
pk_no2 = di['no2'].idxmax()
r = clean[['nox', 'o3']].corr().iloc[0, 1]
summary = f"""花蓮站（環境部一般測站，位於花蓮市區）{clean['datacreationdate'].min():%Y/%m/%d}～{clean['datacreationdate'].max():%Y/%m/%d} 逐時資料，共 {len(clean)} 筆。
近 30 天平均日變化：O3 在 {pk_o3} 點最高（{di['o3'].max()} ppb），{lo_o3} 點最低（{di['o3'].min()} ppb）；
NO2 在 {pk_no2} 點最高（{di['no2'].max()} ppb），早上 8–10 點另有一個小高峰（約 {di.loc[8:10,'no2'].max()} ppb）；
NOx 全天平均只有 {clean['nox'].mean():.1f} ppb；NOx 與 O3 的相關係數 r = {r:.2f}；
風速中午最大（約 {di.loc[12:14,'windspeed'].max()} m/s）。"""
print(summary)

花蓮站（環境部一般測站，位於花蓮市區）2026/09/17～2026/10/17 逐時資料，共 718 筆。
近 30 天平均日變化：O3 在 13 點最高（46.7 ppb），7 點最低（19.2 ppb）；
NO2 在 19 點最高（4.2 ppb），早上 8–10 點另有一個小高峰（約 3.2 ppb）；
NOx 全天平均只有 3.6 ppb；NOx 與 O3 的相關係數 r = 0.06；
風速中午最大（約 3.0 m/s）。


## 步驟 2｜請 AI 從環境化學角度推論成因
把上面的摘要（或圖）貼給 **Gemini／ChatGPT**，用這個 Prompt：
```
這是我撈取花蓮測站的數據摘要（貼上）。我發現 O3 在中午 12–14 點達到峰值，
而 NO2 在晚上 7–9 點最高。請從環境化學、大氣光化學反應機制的角度，幫我解釋為什麼會出現這種現象？
請把你的解釋分成「條列的主張」，每一條都寫出相關的化學反應式，並註明你的依據。
```
⚠️ 申請書的範例寫「NOx 在早上 8 點最高」——**請用你自己的真實數據**改寫 Prompt，不要照抄範例的數字！

## 步驟 3｜查證工具箱
| 查證對象 | 去哪裡查 | 怎麼用 |
|---|---|---|
| 化學反應式、機制 | 課本 Manahan《Environmental Chemistry》第 8、10、11、12 章（中譯本對應章） | 抄下頁碼與原句 |
| 測站環境背景 | 環境部「空氣品質監測網」測站介紹、下面的測站基本資料 | 測站類型、位置、周邊 |
| 文獻 | Google Scholar：Hualien ozone diurnal、NOx titration | 至少 1 篇 |
| 官方指引 | NotebookLM 匯入環境部空品相關官方文件 | 只根據來源回答 |

In [4]:
#@title ▶ 執行：查花蓮測站的基本資料（測站環境背景）
API_KEY = None
try:
    from google.colab import userdata
    API_KEY = userdata.get('MOENV_API_KEY')
except Exception:
    import os; API_KEY = os.environ.get('MOENV_API_KEY')
if API_KEY:
    site = ma.fetch('aqx_p_07', API_KEY, limit=1000)
    print(site[site['sitename'] == '花蓮'][['sitename', 'siteengname', 'areaname', 'township', 'siteaddress', 'sitetype']].to_string(index=False))
else:
    print('沒有金鑰也沒關係：花蓮站＝一般站，位於花蓮市中正路（市區），屬花東空品區。')

sitename siteengname areaname township siteaddress sitetype
      花蓮     Hualien    花東空品區      花蓮市  花蓮市中正路210號      一般站


## 步驟 4｜AI 推論查證表（每條主張一列，寫進學習單與報告書「AI 揪錯與查證」專章）
| # | AI 說的一句關鍵主張 | 判定（✔／△／✘） | 我的查證依據（課本頁數／官方資料／文獻） | 正確說法應該是 |
|---|---|---|---|---|
| 1 |  |  |  |  |
| 2 |  |  |  |  |
| 3 |  |  |  |  |

## 步驟 5｜防線與反思
1. AI 哪一句**最像真的但其實是錯的**？你是怎麼發現的？
2. 這份數據只有**一個測站、30 天**：哪些結論**不能**推論到「整個花蓮」？（數據偏誤）
3. 如果環保局只根據 AI 的解釋做政策，誰可能受影響？（環境正義）
4. 在環境科學研究中，我們該如何**安全且負責任**地使用生成式 AI？

➡️ 課後學習單：[第 16 週 模組活動三](https://rlirdo.github.io/envchem-ai-module-kit/worksheets/W16_M3.html)　｜　報告書範本：[花蓮在地環境數據 AI 協作解讀報告書](https://rlirdo.github.io/envchem-ai-module-kit/report.html)

<sub>NDHU 自資系 游豐兆 博士生製作｜指導教授 楊悠娟 副教授｜國立東華大學 115-1 環境化學 × 生成式 AI 融入專業課程計畫</sub>